In [44]:
import pandas as pd 
import numpy as np 
import matplotlib 
from pathlib import Path

# Model Selection and Evaluation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# for hyperparam tuning
from sklearn.model_selection import GridSearchCV

In [45]:
data_path = Path('../data/preprocessed_pokemon_data.csv')
pkmn_df = pd.read_csv(data_path)
seed = 42

In [49]:
pkmn_df['name-clean']['charmander']

KeyError: 'name-clean'

In [46]:
target = ['tier']
X = pkmn_df.drop(columns=[t for t in target])
y = pkmn_df[target]

# 80% train 20% test, stratify is to make sure the tier distribution is more even in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)

In [47]:
'''
NOTE: objective: multi-softclass essentially tells the model it is doing a classification problem
given multiple classes like OU, UU, etc. choose the most fitting one and output; softmax converts tier scores->probabilities

'''

tier_model = XGBClassifier(
    n_estimators = 500,
    learning_rate = 0.1,
    max_depth=6,
    num_class=len(np.unique(y))
)

tier_model.fit(X_train, y_train)
predictions = tier_model.predict(X_test)


print(classification_report(y_test, predictions, zero_division=0))

mae = mean_absolute_error(y_test, predictions)


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:name: object, name_clean: object

In [39]:
tier_rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15, 
    max_features='sqrt',
    random_state=seed,
    class_weight='balanced_subsample' # chose because some tiers have very few mons
)

tier_rf.fit(X_train, y_train.values.ravel())
rf_predictions = tier_rf.predict(X_test)

# 4. EVALUATE
print("--- Random Forest Tier Prediction Results ---")
print(classification_report(y_test, rf_predictions, zero_division=0))

--- Random Forest Tier Prediction Results ---
              precision    recall  f1-score   support

           0       0.92      0.99      0.95        96
           1       0.90      0.60      0.72        45
           2       0.75      0.94      0.84       140
           3       0.25      0.06      0.10        17
           4       0.50      0.20      0.29        20
           5       0.71      0.63      0.67        19

    accuracy                           0.80       337
   macro avg       0.67      0.57      0.59       337
weighted avg       0.78      0.80      0.78       337



In [ ]:
# tier mapping for each number
tier_order = ['LC', 'NFE', 'RU', 'UU', 'OU', 'Uber']
print(list(zip(tier_order, [i for i in range(10)])))

[('LC', 0), ('NFE', 1), ('RU', 2), ('UU', 3), ('OU', 4), ('Uber', 5)]
